# RAG Assignment: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Document:** FIDE Laws of Chess (effective 1 January 2023), 41 pages — `data/document.pdf`

This notebook builds two retrieval pipelines over the **same chunked corpus** and compares them:

| | Pipeline 1 — Baseline | Pipeline 2 — Re-Ranking |
|---|---|---|
| Stage 1 | Bi-encoder (`all-MiniLM-L6-v2`) → FAISS inner product → **Top 10** | *same Top 10* |
| Stage 2 | take ranks 1–3 as they are | score all 10 (query, chunk) pairs with Cross-Encoder (`ms-marco-MiniLM-L-6-v2`) → re-sort → **Top 3** |
| Generation | Gemini (`gemini-3.8-flash`) → **Answer 1** | same model, **identical prompt** → **Answer 2** |

**Sections**
1. Setup & configuration
2. Load and clean the PDF
3. Clause-aware chunking
4. Bi-encoder index (FAISS)
5. Cross-encoder and LLM
6. The two pipelines
7. Sanity check on a normal query
8. Failure-mode stress tests (keyword traps, exceptions, negation, buried chunks)
9. Rank-shift analysis
10. LLM Answer 1 vs. Answer 2
11. Aggregate retrieval metrics (Hit@3, MRR)
12. Latency profiling
13. Scaling experiment: cross-encoder over the whole corpus
14. Save results

## 1. Setup & configuration

In [1]:
# Colab runtime only: install the packages Colab does not ship. On a local kernel this cell does nothing.
import os, sys
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    %pip install -q pymupdf faiss-cpu python-dotenv google-genai
print("Running on Colab" if IN_COLAB else "Running locally")

Running on Colab


In [2]:
# Standard library
import os, re, time, json, statistics
from pathlib import Path

# Third-party
import numpy as np
import pandas as pd
import pymupdf                      # PDF text extraction (PyMuPDF)
import faiss                        # vector index
from sentence_transformers import SentenceTransformer, CrossEncoder
from google import genai
from google.genai import types
from dotenv import load_dotenv
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

# ---------------- Configuration ----------------
PDF_PATH        = Path("data/document.pdf")
if not PDF_PATH.exists() and Path("document.pdf").exists():   # uploaded to the Colab root instead
    PDF_PATH = Path("document.pdf")
assert PDF_PATH.exists(), "PDF missing: upload data/document.pdf to the runtime (Colab: Upload to Colab)"
RESULTS_DIR     = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)

BI_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"   # dense bi-encoder (384-dim)
CROSS_ENC_NAME  = "cross-encoder/ms-marco-MiniLM-L-6-v2"     # re-ranker
LLM_NAME        = "gemini-3.8-flash"   # free tier of gemini-2.5-flash is only 20 requests/day

TOP_K = 10            # candidates retrieved by the bi-encoder (both pipelines share these)
TOP_N = 3             # chunks passed to the LLM
MAX_CHUNK_CHARS = 700 # soft size limit for one chunk
LLM_RPM = 5           # Gemini free tier: max requests per minute (calls are spaced out to stay under it)

# API key is read from .env (GEMINI_API_KEY=...); on Colab there is no .env, so ask for it once
load_dotenv()
if not os.getenv("GEMINI_API_KEY"):
    from getpass import getpass
    os.environ["GEMINI_API_KEY"] = getpass("GEMINI_API_KEY: ")
print("Config OK")

Config OK


## 2. Load and clean the PDF

PyMuPDF extracts the text page by page. Every page starts with a repeated running header
("Laws of Chess / FIDE Laws of Chess taking effect…") which carries no information, so it is removed.
Each remaining line keeps its page number so chunks can be traced back to the source.

In [3]:
doc = pymupdf.open(PDF_PATH)
print(f"Pages: {doc.page_count}")

HEADER_PREFIXES = ("Laws of Chess / FIDE", "FIDE Laws of Chess taking effect")

lines = []   # list of (page_number, line_text)
for page_no, page in enumerate(doc, start=1):
    for line in page.get_text().split("\n"):
        line = line.strip()
        if not line or line.startswith(HEADER_PREFIXES):
            continue
        lines.append((page_no, line))

print(f"Non-empty lines after cleaning: {len(lines)}")
print("\n".join(l for _, l in lines[:12]))

Pages: 41
Non-empty lines after cleaning: 1300
FIDE LAWS OF CHESS
Approved by the FIDE General Assembly on 07/08/2022
Applied from 01/01/2023
0.1     Introduction
FIDE Laws of Chess cover over-the-board play.
The Laws of Chess have two parts: 1. Basic Rules of Play and 2. Competitive Rules of Play.
The English text is the authentic version of the Laws of Chess (which were adopted at the 93rd
FIDE Congress at Chennai, India) coming into force on 1 January 2023.
0.2     Preface
The Laws of Chess cannot cover all possible situations that may arise during a game, nor can
they regulate all administrative questions. Where cases are not precisely regulated by an Article
of the Laws, it should be possible to reach a correct decision by studying analogous situations


## 3. Clause-aware chunking

The Laws are organised as numbered clauses (`4.3.1`, `7.5.5`, `A.3`, `III.3.1` …).
Fixed-size character windows would cut clauses in half and mix unrelated rules, so the chunker:

1. **Splits the text into clause units**: a new unit starts on every line that begins with a clause number,
   or on a section heading (`Article 7: …`, `Appendix A.`, `Guidelines III.`, `Glossary`).
2. **Packs consecutive units** of the same section into one chunk until ~700 characters,
   so short clauses (e.g. `9.2.1`, `9.2.2`) stay together with their siblings.
3. **Never crosses a section boundary**, so every chunk belongs to exactly one Article/Appendix.
4. Splits anything still too long (the glossary table) on word boundaries.

Each chunk keeps metadata: section, page, and the list of clause numbers it contains. The clause list is
what we later use as **ground truth** ("the answer lives in clause 6.9").

In [4]:
CLAUSE_RE  = re.compile(r"^((?:\d{1,2}|[A-D]|I{1,3})\.\d+(?:\.\d+)*)\s+\S")   # e.g. 4.3.1  A.5.2  III.3.1
HEADING_RE = re.compile(r"^(Article \d+:|Appendix [A-D]\.|Guidelines I{1,3}\.|Glossary of Terms"
                        r"|BASIC RULES|COMPETITIVE RULES|APPENDICES|GUIDELINES)")

# ---- Step 1: clause units ----
units, section, cur = [], "Front matter", None
for page_no, line in lines:
    if HEADING_RE.match(line):
        if not line.isupper():                 # all-caps lines are part headings, keep the Article name
            section = line
        if cur: units.append(cur); cur = None
        continue
    m = CLAUSE_RE.match(line)
    # start a new unit on a clause number, or on a new page for un-numbered text (glossary)
    new_page_free_text = cur is not None and not cur["clause"] and page_no != cur["page"]
    if m or cur is None or new_page_free_text:
        if cur: units.append(cur)
        cur = {"section": section, "page": page_no, "clause": m.group(1) if m else "", "text": line}
    else:
        cur["text"] += " " + line              # continuation line of the same clause
if cur: units.append(cur)

# ---- Step 2 + 3: pack units into chunks within one section ----
packed = []
for u in units:
    last = packed[-1] if packed else None
    if last and last["section"] == u["section"] and len(last["text"]) + len(u["text"]) <= MAX_CHUNK_CHARS:
        last["text"] += "\n" + u["text"]
        if u["clause"]: last["clauses"].append(u["clause"])
    else:
        packed.append({"section": u["section"], "page": u["page"],
                       "clauses": [u["clause"]] if u["clause"] else [], "text": u["text"]})

# ---- Step 4: split oversize chunks on word boundaries ----
chunks = []
for c in packed:
    text = c["text"]
    while len(text) > MAX_CHUNK_CHARS * 1.3:
        cut = text.rfind(" ", 0, MAX_CHUNK_CHARS)
        chunks.append({**c, "text": text[:cut]})
        text = text[cut:].strip()
    chunks.append({**c, "text": text})

def chunk_label(c):
    # Human-readable label, e.g. "7.5.4–7.5.5" or "Glossary p.36"
    if c["clauses"]:
        return c["clauses"][0] if len(c["clauses"]) == 1 else f'{c["clauses"][0]}–{c["clauses"][-1]}'
    return f'{c["section"].split(":")[0].split(" of ")[0]} p.{c["page"]}'

for i, c in enumerate(chunks):
    c["id"] = i
    c["label"] = chunk_label(c)

lengths = [len(c["text"]) for c in chunks]
print(f"{len(units)} clause units  ->  {len(chunks)} chunks")
print(f"chunk length (chars): min {min(lengths)}, mean {np.mean(lengths):.0f}, max {max(lengths)}")

chunk_df = pd.DataFrame([{"id": c["id"], "label": c["label"], "page": c["page"],
                          "section": c["section"][:45], "chars": len(c["text"])} for c in chunks])
display(chunk_df.iloc[[0, 13, 18, 33, 39, 66, 104, len(chunks)-1]])

372 clause units  ->  124 chunks
chunk length (chars): min 94, mean 570, max 867


,id,label,page,section,chars
0,0,0.1,1,Front matter,417
13,13,3.8.2.1–3.9,8,Article 3: The Moves of the Pieces,459
18,18,4.4.2–4.4.4,9,Article 4: The Act of Moving the Pieces,571
33,33,6.9–6.10.1,12,Article 6: The Chessclock,662
39,39,7.5.2–7.5.4,14,Article 7: Irregularities,546
66,66,A.1–A.4,21,Appendix A. Rapid Chess,691
104,104,III.1–III.3.1.1,33,Guidelines III. Games without Increment inclu,626
123,123,Glossary p.41,41,Glossary of Terms in the Laws of Chess,787


In [5]:
# Look at one full chunk to check the formatting is clean
print(chunks[39]["label"], "|", chunks[39]["section"], "| page", chunks[39]["page"])
print("-" * 80)
print(chunks[39]["text"])

7.5.2–7.5.4 | Article 7: Irregularities | page 14
--------------------------------------------------------------------------------
7.5.2    If the player has moved a pawn to the furthest distant rank, pressed the clock, but not replaced the pawn with a new piece, the move is illegal. The pawn shall be replaced by a queen of the same colour as the pawn.
7.5.3    If the player presses the clock without making a move, it shall be considered and penalised as if an illegal move.
7.5.4    If a player uses two hands to make a single move (for example in case of castling, capturing or promotion) and pressed the clock, it shall be considered and penalised as if an illegal move.


## 4. Bi-encoder index (FAISS)

The bi-encoder maps every chunk to a 384-dim vector **independently** of any query.
Vectors are L2-normalised, so the inner product in `IndexFlatIP` equals **cosine similarity**.
`IndexFlatIP` is exact (brute-force) search, fine for ~120 chunks.

In [6]:
bi_encoder = SentenceTransformer(BI_ENCODER_NAME)

t0 = time.perf_counter()
chunk_embeddings = bi_encoder.encode([c["text"] for c in chunks], batch_size=32,
                                     normalize_embeddings=True, show_progress_bar=False)
index_time = time.perf_counter() - t0

index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings.astype("float32"))
print(f"Indexed {index.ntotal} chunks, dim={chunk_embeddings.shape[1]}, offline encoding took {index_time:.2f}s")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexed 124 chunks, dim=384, offline encoding took 11.91s


## 5. Cross-encoder and LLM

**Cross-encoder:** takes the query and one chunk *together* as one input
(`[CLS] query [SEP] chunk [SEP]`) and outputs one relevance score. It cannot be pre-computed,
it has to run once per (query, chunk) pair at query time.

**LLM:** Gemini 3.8 Flash with `temperature=0` and thinking disabled, so answers are as deterministic as
possible and the only difference between Answer 1 and Answer 2 is the **context chunks**.

In [7]:
cross_encoder = CrossEncoder(CROSS_ENC_NAME)
llm = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

GEN_CONFIG = types.GenerateContentConfig(
    temperature=0.0,
    thinking_config=types.ThinkingConfig(thinking_budget=0),
)

# One prompt template used by BOTH pipelines. Only {context} changes.
PROMPT_TEMPLATE = '''You are an expert on the FIDE Laws of Chess.
Answer the question using ONLY the context excerpts below.
If the context does not contain the answer, say "The provided context does not contain this information."
Cite the article numbers you rely on. Keep the answer under 120 words.

Context:
{context}

Question: {question}

Answer:'''

def build_prompt(question, context_chunks):
    context = "\n\n".join(f"[Excerpt {i+1} | {c['section']} | clauses {c['label']}]\n{c['text']}"
                          for i, c in enumerate(context_chunks))
    return PROMPT_TEMPLATE.format(context=context, question=question)

# Answers are cached on disk keyed by (model, prompt): re-running the notebook does not spend API
# quota again. Delete results/llm_cache.json to force fresh generations.
CACHE_PATH = RESULTS_DIR / "llm_cache.json"
llm_cache = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}
LAST_CALL = {"cached": False, "t_api": 0.0}

def generate(prompt, retries=4):
    key = f"{LLM_NAME}::{prompt}"
    if key in llm_cache:
        LAST_CALL["cached"] = True
        return llm_cache[key]["answer"]
    LAST_CALL["cached"] = False
    for attempt in range(retries):
        # wait so that API calls are at least 60/LLM_RPM seconds apart (per-minute rate limit)
        wait = LAST_CALL["t_api"] + 60 / LLM_RPM + 1 - time.perf_counter()
        if wait > 0:
            time.sleep(wait)
        LAST_CALL["t_api"] = time.perf_counter()
        try:
            t0 = time.perf_counter()
            answer = llm.models.generate_content(model=LLM_NAME, contents=prompt, config=GEN_CONFIG).text.strip()
            llm_cache[key] = {"answer": answer, "latency_s": time.perf_counter() - t0}
            CACHE_PATH.write_text(json.dumps(llm_cache, indent=1))
            return answer
        except Exception as e:
            msg = str(e)
            # A daily free-tier quota will not recover by waiting a few seconds: stop with a clear error
            if "RESOURCE_EXHAUSTED" in msg and re.search(r"retry in \d+h", msg):
                raise RuntimeError(f"Daily Gemini quota exhausted for {LLM_NAME}. Change LLM_NAME or wait.") from e
            wait = 60 if "RESOURCE_EXHAUSTED" in msg else 2 ** attempt * 5   # per-minute limit: wait for the window to reset
            print(f"  LLM error ({msg[:80]}...), retrying in {wait}s")
            time.sleep(wait)
    return "[generation failed]"

def llm_latency(prompt):
    # Measured API latency of the call that produced this answer (also valid when loaded from cache)
    return llm_cache.get(f"{LLM_NAME}::{prompt}", {}).get("latency_s", float("nan"))

# warm-up: the first encoder calls are slower (lazy init), so run them once before any timing
_ = cross_encoder.predict([("warm up", "warm up")]); _ = bi_encoder.encode(["warm up"])
print(f"Cross-encoder ready, LLM = {LLM_NAME}, cached answers: {len(llm_cache)}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Cross-encoder ready, LLM = gemini-3.8-flash, cached answers: 22


## 6. The two pipelines

* `retrieve_top_k` is **shared**: both pipelines start from exactly the same 10 candidates.
* `pipeline_baseline` keeps the bi-encoder order and sends ranks 1–3 to the LLM.
* `pipeline_rerank` scores the same 10 candidates with the cross-encoder, re-sorts them, and sends the new top 3.

Each function returns the chunks, scores, and **timings** for every stage (`time.perf_counter`).

In [8]:
def retrieve_top_k(query, k=TOP_K):
    # Stage 1 (shared): embed query -> FAISS inner-product search
    t0 = time.perf_counter()
    q_emb = bi_encoder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q_emb, k)
    elapsed = time.perf_counter() - t0
    candidates = [{"chunk": chunks[i], "bi_rank": r + 1, "bi_score": float(s)}
                  for r, (i, s) in enumerate(zip(ids[0], scores[0]))]
    return candidates, elapsed

def rerank(query, candidates):
    # Stage 2: cross-encoder scores every (query, chunk) pair jointly, then re-sort descending
    t0 = time.perf_counter()
    ce_scores = cross_encoder.predict([(query, c["chunk"]["text"]) for c in candidates])
    elapsed = time.perf_counter() - t0
    for c, s in zip(candidates, ce_scores):
        c["ce_score"] = float(s)
    reranked = sorted(candidates, key=lambda c: c["ce_score"], reverse=True)
    for r, c in enumerate(reranked):
        c["ce_rank"] = r + 1
    return reranked, elapsed

def pipeline_baseline(query, with_llm=True):
    candidates, t_retr = retrieve_top_k(query)
    top = [c["chunk"] for c in candidates[:TOP_N]]
    answer, t_llm = None, 0.0
    if with_llm:
        t0 = time.perf_counter(); answer = generate(build_prompt(query, top)); t_llm = time.perf_counter() - t0
    return {"candidates": candidates, "context": top, "answer": answer,
            "t_retrieve": t_retr, "t_rerank": 0.0, "t_llm": t_llm}

def pipeline_rerank(query, with_llm=True, candidates=None):
    # `candidates` lets us reuse the exact Top-10 from the baseline run (same corpus, same candidates)
    t_retr = 0.0
    if candidates is None:
        candidates, t_retr = retrieve_top_k(query)
    candidates = [dict(c) for c in candidates]          # copy, don't mutate the baseline result
    reranked, t_rr = rerank(query, candidates)
    top = [c["chunk"] for c in reranked[:TOP_N]]
    answer, t_llm = None, 0.0
    if with_llm:
        t0 = time.perf_counter(); answer = generate(build_prompt(query, top)); t_llm = time.perf_counter() - t0
    return {"candidates": reranked, "context": top, "answer": answer,
            "t_retrieve": t_retr, "t_rerank": t_rr, "t_llm": t_llm}

def candidates_table(reranked):
    # Top-10 table showing both rankings side by side, in baseline order
    rows = sorted(reranked, key=lambda c: c["bi_rank"])
    return pd.DataFrame([{
        "bi_rank": c["bi_rank"], "ce_rank": c["ce_rank"],
        "shift": c["bi_rank"] - c["ce_rank"],
        "cosine": round(c["bi_score"], 3), "ce_score": round(c["ce_score"], 2),
        "clauses": c["chunk"]["label"], "text": c["chunk"]["text"][:90].replace("\n", " ") + "…",
    } for c in rows])

## 7. Sanity check on a normal query
A straightforward question first, to confirm both pipelines work end to end.

In [9]:
q = "How does the knight move?"
b = pipeline_baseline(q)
r = pipeline_rerank(q, candidates=b["candidates"])
display(candidates_table(r["candidates"]))
print("Answer 1 (baseline):", b["answer"])
print("\nAnswer 2 (re-rank): ", r["answer"])

,bi_rank,ce_rank,shift,cosine,ce_score,clauses,text
0,1,7,-6,0.560,-1.76,C.10.2.2–C.10.3.3,"C.10.2.2 The rank of the square of departure, and C.10.2.3 The square of arrival. C.…"
1,2,1,1,0.554,5.53,3.3–3.7.1,3.3 The rook may move to any square along the file or the rank on which it stands. 3.4…
2,3,4,-1,0.553,-0.32,3.8.2,3.8.2 by ‘castling’. This is a move of the king and either rook of the same colour alon…
3,4,3,1,0.530,1.96,Glossary p.38,"j’adoube 4.2 Giving notice that the player wishes to adjust a piece, but does not necessar…"
4,5,5,0,0.503,-0.56,1.1–1.4,1.1 The game of chess is played between two opponents who move their pieces on a squar…
5,6,2,4,0.476,2.25,3.7.3.3–3.8.1,"3.7.3.3 When a player, having the move, plays a pawn to the rank furthest from its star…"
6,7,6,1,0.442,-0.91,II.3.2–II.3.2.4,"II.3.2 How to castle. In Chess960, depending on the pre-castling position of the castli…"
7,8,8,0,0.437,-3.00,3.1–3.2,3.1 It is not permitted to move a piece to a square occupied by a piece of the same co…
8,9,10,-1,0.425,-3.52,9.2.3–9.3,"9.2.3 Positions are considered the same if and only if the same player has the move, pi…"
9,10,9,1,0.425,-3.37,3.7.2–3.7.3.2,3.7.2 on its first move the pawn may move as in 3.7.1 or alternatively it may advance t…


Answer 1 (baseline): According to Article 3.6, the knight may move to one of the squares nearest to that on which it stands, but not on the same rank, file, or diagonal.

Answer 2 (re-rank):  According to Article 3.6, the knight may move to one of the squares nearest to that on which it stands, but not on the same rank, file, or diagonal.


## 8. Failure-mode stress tests

The queries below were designed for this document, then **checked against the actual retrieval output**.
Each one has a **gold clause**: the clause that holds the answer.
A query is a "baseline failure" when the gold chunk is in the Top 10 but **not in the baseline Top 3**.

| Category | What goes wrong for the bi-encoder |
|---|---|
| **Keyword trap** | the query shares words ("clock", "castle") with many irrelevant clauses, while the answer uses different words ("pause the chessclock", "two hands … castling") |
| **Exception / corner case** | the general rule is retrieved, the "except…/however…" clause or the special-case appendix is missed |
| **Negation / constraint** | "not allowed", "must not": embeddings of "may" and "may not" sentences are almost identical |
| **Buried chunk** | the right chunk sits at rank 4–10 for no single obvious reason |

One **control** query is included where the cross-encoder makes things *worse*, to show re-ranking is not always a win.

In [10]:
FAILURE_QUERIES = [
    # (id, category, query, gold clause(s), why it is hard)
    ("KT1", "Keyword trap", "Can a player stop the clock to go to the toilet?", ["6.11.2"],
     "'clock' pulls in all clock-handling clauses (6.2.x); the answer says 'pause the chessclock only to seek the arbiter's assistance'"),
    ("KT2", "Keyword trap", "Can a player use two hands to castle?", ["7.5.4"],
     "'castle' pulls in the castling-rules clauses (3.8.2, 4.4); the penalty clause is in Article 7 (Irregularities)"),
    ("EX1", "Exception", "If a player runs out of time, do they always lose the game?", ["6.9"],
     "general flag/default-time rules are retrieved; the 'however, the game is drawn if the opponent cannot checkmate' exception is 6.9"),
    ("EX2", "Exception", "Is the draw automatic after 50 moves without capture?", ["9.6.2"],
     "the 50-move CLAIM rule (9.3) is retrieved; the AUTOMATIC draw only happens at 75 moves (9.6.2)"),
    ("NG1", "Negation", "What is the arbiter not allowed to tell a player during the game?", ["12.6"],
     "'not allowed' matches permission clauses (11.2.x); 12.6 says the arbiter 'must not' / 'shall refrain'"),
    ("BU1", "Buried chunk", "Can a player be penalised for knocking pieces over?", ["7.4.3"],
     "'knocking over' is only used for the clock (6.2.4); the rule for pieces says 'displaces'"),
    ("CTRL", "Control (regression)", "Can the time on the clock be corrected after an illegal move?", ["7.1"],
     "a case where the cross-encoder pushes the correct chunk DOWN"),
]

def gold_chunk_ids(gold_clauses):
    # Chunks that contain any gold clause (exact clause match or a sub-clause of it, e.g. III.3.1 -> III.3.1.1)
    return {c["id"] for c in chunks for g in gold_clauses
            for cl in c["clauses"] if cl == g or cl.startswith(g + ".")}

for qid, cat, q, gold, _ in FAILURE_QUERIES:
    print(f"{qid:5s} gold {gold} -> chunk ids {sorted(gold_chunk_ids(gold))}")

KT1   gold ['6.11.2'] -> chunk ids [34]
KT2   gold ['7.5.4'] -> chunk ids [39]
EX1   gold ['6.9'] -> chunk ids [33]
EX2   gold ['9.6.2'] -> chunk ids [54]
NG1   gold ['12.6'] -> chunk ids [63]
BU1   gold ['7.4.3'] -> chunk ids [37]
CTRL  gold ['7.1'] -> chunk ids [36]


## 9. Rank-shift analysis

For every query: run Stage 1 once, then re-rank **the same** 10 candidates, and record where the gold chunk
ended up in each ranking.

In [11]:
def best_rank(candidates, gold_ids, key):
    ranks = [c[key] for c in candidates if c["chunk"]["id"] in gold_ids]
    return min(ranks) if ranks else None     # None = not in the Top 10 at all

runs = {}          # qid -> (baseline_result, rerank_result)
shift_rows = []
for qid, cat, q, gold, why in FAILURE_QUERIES:
    b = pipeline_baseline(q, with_llm=False)
    r = pipeline_rerank(q, with_llm=False, candidates=b["candidates"])
    runs[qid] = (b, r)
    g = gold_chunk_ids(gold)
    br, cr = best_rank(r["candidates"], g, "bi_rank"), best_rank(r["candidates"], g, "ce_rank")
    gold_label = next(c["chunk"]["label"] for c in r["candidates"] if c["chunk"]["id"] in g) if br else "-"
    shift_rows.append({
        "id": qid, "category": cat, "query": q, "gold clause": ", ".join(gold), "gold chunk": gold_label,
        "baseline rank": br, "re-ranked rank": cr,
        "rank shift": f"#{br} → #{cr}" if br else "not in top 10",
        "in baseline top3": bool(br and br <= TOP_N), "in re-ranked top3": bool(cr and cr <= TOP_N),
        "baseline top3": " | ".join(c["label"] for c in b["context"]),
        "re-ranked top3": " | ".join(c["label"] for c in r["context"]),
    })

shift_df = pd.DataFrame(shift_rows)
display(shift_df[["id", "category", "gold clause", "gold chunk", "rank shift",
                  "in baseline top3", "in re-ranked top3", "baseline top3", "re-ranked top3"]])

,id,category,gold clause,gold chunk,rank shift,in baseline top3,in re-ranked top3,baseline top3,re-ranked top3
0,KT1,Keyword trap,6.11.2,6.10.2–6.11.3,#8 → #1,False,True,6.2.1.2–6.2.3 | 6.2.4–6.3 | 8.4–8.5.1,6.10.2–6.11.3 | 6.2.1.2–6.2.3 | 6.2.4–6.3
1,KT2,Keyword trap,7.5.4,7.5.2–7.5.4,#8 → #1,False,True,4.4.2–4.4.4 | 3.8.2.1–3.9 | 4.3.2–4.4.1,7.5.2–7.5.4 | II.2.3–II.3.1 | 4.4.2–4.4.4
2,EX1,Exception,6.9,6.9–6.10.1,#7 → #2,False,True,6.7.2–6.8 | 8.6–8.7 | 8.4–8.5.1,I.10–I.12 | 6.9–6.10.1 | Glossary p.37
3,EX2,Exception,9.6.2,9.6.2,#4 → #3,False,True,9.3.1–9.5 | III.3.1.2–III.4 | 9.1.2.3–9.2.2,9.3.1–9.5 | Glossary p.41 | 9.6.2
4,NG1,Negation,12.6,12.4–12.6,#4 → #3,False,True,11.2.4–11.3.2 | 12.7–12.9.3 | 11.3.3–11.4,12.7–12.9.3 | 11.2.4–11.3.2 | 12.4–12.6
5,BU1,Buried chunk,7.4.3,7.3–7.5,#8 → #3,False,True,11.5–11.8 | 4.8–4.9 | 12.9.4–12.9.9,6.2.4–6.3 | 4.8–4.9 | 7.3–7.5
6,CTRL,Control (regression),7.1,7.1–7.2.2,#3 → #7,True,False,7.5.1 | 7.5.2–7.5.4 | 7.1–7.2.2,7.5.1 | I.13–I.14 | 7.5.2–7.5.4


### Detailed Top-10 tables
Each table is in **baseline order**. `shift` > 0 means the cross-encoder moved the chunk up. The gold chunk is marked with ★.

In [12]:
for qid, cat, q, gold, why in FAILURE_QUERIES:
    b, r = runs[qid]
    g = gold_chunk_ids(gold)
    t = candidates_table(r["candidates"])
    t.insert(0, "gold", ["★" if c["chunk"]["id"] in g else "" for c in sorted(r["candidates"], key=lambda c: c["bi_rank"])])
    display(Markdown(f"**{qid} · {cat}** — *{q}*  \nGold clause **{', '.join(gold)}**. Why it is hard: {why}"))
    display(t)

**KT1 · Keyword trap** — *Can a player stop the clock to go to the toilet?*  
Gold clause **6.11.2**. Why it is hard: 'clock' pulls in all clock-handling clauses (6.2.x); the answer says 'pause the chessclock only to seek the arbiter's assistance'

,gold,bi_rank,ce_rank,shift,cosine,ce_score,clauses,text
0,,1,2,-1,0.500,-1.43,6.2.1.2–6.2.3,"6.2.1.2 the player has made his/her next move, when his/her previous move was not comp…"
1,,2,3,-1,0.486,-1.86,6.2.4–6.3,6.2.4 The players must handle the chessclock properly. It is forbidden to press it forc…
2,,3,4,-1,0.448,-4.10,8.4–8.5.1,8.4 If a player has less than five minutes left on his/her clock during an allotted pe…
3,,4,6,-2,0.437,-5.75,8.1.6–8.3,"8.1.6 If a player is unable to keep score, an assistant, who must be acceptable to the …"
4,,5,5,0,0.432,-5.01,6.11.4–6.12.2,"6.11.4 If a player pauses the chessclock in order to seek the arbiter’s assistance, the…"
5,,6,10,-4,0.421,-9.16,D.2.6.1–D.2.6.2,D.2.6.1 A specially constructed chessclock for the visually disabled shall be admissibl…
6,,7,7,0,0.409,-8.07,6.3.2–6.4,6.3.2 The time saved by a player during one period is added to his/her time available f…
7,★,8,1,7,0.408,-1.40,6.10.2–6.11.3,6.10.2 If during a game it is found that the setting of either or both clocks is incorr…
8,,9,9,0,0.406,-8.98,D.2.10.1–D.2.10.6,"D.2.10.1 making either player's move on the board of the opponent, D.2.10.2 announci…"
9,,10,8,2,0.400,-8.73,A.4.1–A.5.1.1,A.4.1 one arbiter supervises at most three games and A.4.2 each game is recorded by …


**KT2 · Keyword trap** — *Can a player use two hands to castle?*  
Gold clause **7.5.4**. Why it is hard: 'castle' pulls in the castling-rules clauses (3.8.2, 4.4); the penalty clause is in Article 7 (Irregularities)

,gold,bi_rank,ce_rank,shift,cosine,ce_score,clauses,text
0,,1,3,-2,0.478,-0.13,4.4.2–4.4.4,4.4.2 deliberately touches a rook and then his/her king he/she is not allowed to castle…
1,,2,7,-5,0.434,-5.62,3.8.2.1–3.9,"3.8.2.1 The right to castle has been lost: 1) If the king has already moved, or 2) With…"
2,,3,4,-1,0.430,-3.03,4.3.2–4.4.1,"4.3.2 one or more of his/her opponent’s pieces, he/she must capture the first piece tou…"
3,,4,5,-1,0.419,-3.66,4.1–4.3.1,4.1 Each move must be played with one hand only. 4.2 Adjusting the pieces or other…
4,,5,2,3,0.378,0.09,II.2.3–II.3.1,II.2.3 the black pieces are placed opposite the white pieces. The starting position can…
5,,6,6,0,0.367,-4.44,II.3.2–II.3.2.4,"II.3.2 How to castle. In Chess960, depending on the pre-castling position of the castli…"
6,,7,10,-3,0.348,-7.73,6.2.1.2–6.2.3,"6.2.1.2 the player has made his/her next move, when his/her previous move was not comp…"
7,★,8,1,7,0.332,0.20,7.5.2–7.5.4,"7.5.2 If the player has moved a pawn to the furthest distant rank, pressed the clock, b…"
8,,9,8,1,0.320,-5.83,4.7.2–4.7.3,"4.7.2 Castling, when the player's hand has released the rook on the square previously c…"
9,,10,9,1,0.317,-6.69,II.3.2.7,"II.3.2.7 Notes 1. To avoid any misunderstanding, it may be useful to state ""I am about …"


**EX1 · Exception** — *If a player runs out of time, do they always lose the game?*  
Gold clause **6.9**. Why it is hard: general flag/default-time rules are retrieved; the 'however, the game is drawn if the opponent cannot checkmate' exception is 6.9

...

,gold,bi_rank,ce_rank,shift,cosine,ce_score,clauses,text
0,,1,2,-1,0.707,6.07,11.2.4–11.3.2,11.2.4 The regulations of an event may specify that the opponent of the player having a…
1,,2,1,1,0.691,6.50,12.7–12.9.3,"12.7 If someone observes an irregularity, he/she may inform only the arbiter. Players in…"
2,,3,4,-1,0.633,4.46,11.3.3–11.4,"11.3.3 The arbiter may require the player to allow his/her clothes, bags, other items …"
3,★,4,3,1,0.633,5.82,12.4–12.6,"12.4 The arbiter may appoint assistants to observe games, for example when several playe…"
4,,5,6,-1,0.602,2.99,11.9–11.12,11.9 A player shall have the right to request from the arbiter an explanation of particu…
5,,6,10,-4,0.578,-0.29,11.5–11.8,11.5 It is forbidden to distract or annoy the opponent in any manner whatsoever. This in…
6,,7,9,-2,0.571,1.02,4.8–4.9,4.8 A player forfeits his/her right to claim against his/her opponent’s violation of A…
7,,8,8,0,0.561,2.32,A.4.1–A.5.1.1,A.4.1 one arbiter supervises at most three games and A.4.2 each game is recorded by …
8,,9,7,2,0.557,2.69,11.3.2.1–11.3.2.2,"11.3.2.1 However, the regulations of an event may allow such devices to be stored in a …"
9,,10,5,5,0.537,3.81,6.11.4–6.12.2,"6.11.4 If a player pauses the chessclock in order to seek the arbiter’s assistance, the…"


**BU1 · Buried chunk** — *Can a player be penalised for knocking pieces over?*  
Gold clause **7.4.3**. Why it is hard: 'knocking over' is only used for the clock (6.2.4); the rule for pieces says 'displaces'

,gold,bi_rank,ce_rank,shift,cosine,ce_score,clauses,text
0,,1,4,-3,0.529,-2.09,11.5–11.8,11.5 It is forbidden to distract or annoy the opponent in any manner whatsoever. This in…
1,,2,2,0,0.494,0.54,4.8–4.9,4.8 A player forfeits his/her right to claim against his/her opponent’s violation of A…
2,,3,8,-5,0.448,-8.94,12.9.4–12.9.9,12.9.4 increasing the points scored in the game by the opponent to the maximum availabl…
3,,4,5,-1,0.429,-4.38,12.7–12.9.3,"12.7 If someone observes an irregularity, he/she may inform only the arbiter. Players in…"
4,,5,9,-4,0.408,-8.96,11.9–11.12,11.9 A player shall have the right to request from the arbiter an explanation of particu…
5,,6,1,5,0.380,5.81,6.2.4–6.3,6.2.4 The players must handle the chessclock properly. It is forbidden to press it forc…
6,,7,7,0,0.373,-8.12,7.5.5,"7.5.5 After the action taken under Article 7.5.1, 7.5.2, 7.5.3 or 7.5.4 for the first c…"
7,★,8,3,5,0.364,0.27,7.3–7.5,"7.3 If a game has started with colours reversed then, if less than 10 moves have been …"
8,,9,10,-1,0.358,-10.42,III.5.2–III.6,"III.5.2 If the arbiter postpones his/her decision, the opponent may be awarded two ext…"
9,,10,6,4,0.356,-7.28,Glossary p.34,Term First Reference Definition adjourn 8.1 Instead of playing the game in one session it …


**CTRL · Control (regression)** — *Can the time on the clock be corrected after an illegal move?*  
Gold clause **7.1**. Why it is hard: a case where the cross-encoder pushes the correct chunk DOWN

,gold,bi_rank,ce_rank,shift,cosine,ce_score,clauses,text
0,,1,1,0,0.680,4.54,7.5.1,7.5.1 An illegal move is completed once the player has pressed his/her clock. If during…
1,,2,3,-1,0.602,1.55,7.5.2–7.5.4,"7.5.2 If the player has moved a pawn to the furthest distant rank, pressed the clock, b…"
2,★,3,7,-4,0.599,-2.29,7.1–7.2.2,7.1 If an irregularity occurs and the pieces have to be restored to a previous positio…
3,,4,2,2,0.557,2.58,I.13–I.14,"I.13 If, upon resumption of the game, either player points out before making his/her fi…"
4,,5,8,-3,0.545,-3.41,6.2.4–6.3,6.2.4 The players must handle the chessclock properly. It is forbidden to press it forc…
5,,6,4,2,0.513,1.47,7.5.5,"7.5.5 After the action taken under Article 7.5.1, 7.5.2, 7.5.3 or 7.5.4 for the first c…"
6,,7,10,-3,0.507,-8.01,8.6–8.7,8.6 If the scoresheets cannot be brought up to date showing that a player has overstep…
7,,8,6,2,0.501,-0.77,I.12.1–I.12.2,"I.12.1 If the envelope containing the sealed move is missing, the game shall continue …"
8,,9,5,4,0.499,0.89,6.2.1.2–6.2.3,"6.2.1.2 the player has made his/her next move, when his/her previous move was not comp…"
9,,10,9,1,0.493,-7.82,6.9–6.10.1,"6.9 Except where one of Articles 5.1.1, 5.1.2, 5.2.1, 5.2.2, 5.2.3 applies, if a playe…"


## 10. LLM Answer 1 vs. Answer 2

Both answers use the identical prompt and model. Only the three context chunks differ.
The **Context** line shows which clauses each pipeline gave to the LLM.

In [13]:
answer_rows = []
for qid, cat, q, gold, why in FAILURE_QUERIES:
    b, r = runs[qid]
    # generate with the already-selected contexts, timing each LLM call
    p1, p2 = build_prompt(q, b["context"]), build_prompt(q, r["context"])
    a1 = generate(p1); a2 = generate(p2)
    b["answer"], b["t_llm"], r["answer"], r["t_llm"] = a1, llm_latency(p1), a2, llm_latency(p2)
    answer_rows.append({"id": qid, "category": cat, "query": q, "gold clause": ", ".join(gold),
                        "baseline context": " | ".join(c["label"] for c in b["context"]),
                        "re-ranked context": " | ".join(c["label"] for c in r["context"]),
                        "answer_1_baseline": a1, "answer_2_rerank": a2})
    display(Markdown(
        f"---\n### {qid} · {cat}\n**Q:** {q}  \n**Gold clause:** {', '.join(gold)}\n\n"
        f"**Answer 1 — Baseline** (context: {answer_rows[-1]['baseline context']})\n\n> {a1}\n\n"
        f"**Answer 2 — Re-ranked** (context: {answer_rows[-1]['re-ranked context']})\n\n> {a2}\n"))

answers_df = pd.DataFrame(answer_rows)

---
### KT1 · Keyword trap
**Q:** Can a player stop the clock to go to the toilet?  
**Gold clause:** 6.11.2

**Answer 1 — Baseline** (context: 6.2.1.2–6.2.3 | 6.2.4–6.3 | 8.4–8.5.1)

> The provided context does not contain this information.

**Answer 2 — Re-ranked** (context: 6.10.2–6.11.3 | 6.2.1.2–6.2.3 | 6.2.4–6.3)

> The provided context does not contain this information. 

According to Article 6.11.2, a player may pause the chessclock only to seek the arbiter’s assistance (for example, when a required promotion piece is not available), and Article 6.10.2 permits stopping it if the clock settings are incorrect; the context does not mention going to the toilet.


---
### KT2 · Keyword trap
**Q:** Can a player use two hands to castle?  
**Gold clause:** 7.5.4

**Answer 1 — Baseline** (context: 4.4.2–4.4.4 | 3.8.2.1–3.9 | 4.3.2–4.4.1)

> The provided context does not contain this information.

**Answer 2 — Re-ranked** (context: 7.5.2–7.5.4 | II.2.3–II.3.1 | 4.4.2–4.4.4)

> Based on the provided context, a player cannot use two hands to castle. According to Article 7.5.4, if a player uses two hands to make a single move (specifically noting castling as an example) and presses the clock, it shall be considered and penalised as an illegal move.


---
### EX1 · Exception
**Q:** If a player runs out of time, do they always lose the game?  
**Gold clause:** 6.9

**Answer 1 — Baseline** (context: 6.7.2–6.8 | 8.6–8.7 | 8.4–8.5.1)

> The provided context does not contain this information.

**Answer 2 — Re-ranked** (context: I.10–I.12 | 6.9–6.10.1 | Glossary p.37)

> No, they do not always lose. 

According to Article 6.9, while a player who does not complete the prescribed number of moves in the allotted time generally loses the game, "the game is drawn if the position is such that the opponent cannot checkmate the player’s king by any possible series of legal moves." Additionally, an exception applies where one of Articles 5.1.1, 5.1.2, 5.2.1, 5.2.2, or 5.2.3 applies. 

Furthermore, in an adjourned game where a player arrives after the default time, "if the sealed move resulted in the conclusion of the game, that conclusion shall still apply" (Article I.10).


---
### EX2 · Exception
**Q:** Is the draw automatic after 50 moves without capture?  
**Gold clause:** 9.6.2

**Answer 1 — Baseline** (context: 9.3.1–9.5 | III.3.1.2–III.4 | 9.1.2.3–9.2.2)

> Based on the provided context, the draw is not automatic. According to Articles 9.1.2.3, 9.3.1, 9.3.2, and 9.4, reaching 50 moves without a capture or pawn move provides the basis for a player to claim an intended or completed draw (which also acts as a draw offer), rather than occurring automatically. Touching a piece can even cause the player to lose the right to claim the draw on that move (Article 9.4).

**Answer 2 — Re-ranked** (context: 9.3.1–9.5 | Glossary p.41 | 9.6.2)

> No, the draw is not automatic after 50 moves without capture. 

According to Article 9.3 and the Glossary entry for the "50-move rule", a player *may claim* a draw if the conditions are met (such as declaring the intention on their scoresheet under Article 9.3.1 or if the last 50 moves by each player have been completed under Article 9.3.2). Furthermore, a player loses the right to claim it if they touch a piece under Article 9.4. 

In contrast, an automatic draw without a capture or pawn move only occurs after 75 moves under Article 9.6.2 (the "75-move rule").


---
### NG1 · Negation
**Q:** What is the arbiter not allowed to tell a player during the game?  
**Gold clause:** 12.6

**Answer 1 — Baseline** (context: 11.2.4–11.3.2 | 12.7–12.9.3 | 11.3.3–11.4)

> The provided context does not contain this information.

**Answer 2 — Re-ranked** (context: 12.7–12.9.3 | 11.2.4–11.3.2 | 12.4–12.6)

> According to Article 12.6, the arbiter must not:
* Indicate the number of moves completed, except when applying Article 8.5 when at least one flag has fallen.
* Inform a player that his/her opponent has completed a move.
* Inform a player that he/she has not pressed his/her clock.


---
### BU1 · Buried chunk
**Q:** Can a player be penalised for knocking pieces over?  
**Gold clause:** 7.4.3

**Answer 1 — Baseline** (context: 11.5–11.8 | 4.8–4.9 | 12.9.4–12.9.9)

> The provided context does not contain this information.

**Answer 2 — Re-ranked** (context: 6.2.4–6.3 | 4.8–4.9 | 7.3–7.5)

> Yes. According to Article 7.4.3, the arbiter may penalise a player who displaces pieces. Furthermore, under Article 7.4.1, if a player displaces one or more pieces, they must re-establish the correct position on their own time.


---
### CTRL · Control (regression)
**Q:** Can the time on the clock be corrected after an illegal move?  
**Gold clause:** 7.1

**Answer 1 — Baseline** (context: 7.5.1 | 7.5.2–7.5.4 | 7.1–7.2.2)

> Yes. According to Article 7.5.1, completing an illegal move requires that the position immediately before the irregularity be reinstated. Under Article 7.1, if an irregularity occurs and the pieces must be restored to a previous position, the arbiter shall use their best judgement to determine the times shown on the chessclock. This authority also includes the right not to change the clock times, as well as the ability to adjust the move-counter if necessary.

**Answer 2 — Re-ranked** (context: 7.5.1 | I.13–I.14 | 7.5.2–7.5.4)

> The provided context does not contain this information. 

While Article 7.5.1 addresses reinstating the position prior to an illegal move, and Guidelines I.13 mentions correcting incorrectly indicated time upon the resumption of an adjourned game, the provided excerpts do not state whether the clock's time can be corrected after an illegal move.


## 11. Aggregate retrieval metrics

The failure queries were chosen *because* they are hard, so they do not show the average effect.
Here both rankings are scored on a wider set of 36 hand-labelled questions (normal questions plus the
failure queries). No LLM is needed for this, only the gold clause.

* **Hit@3**: fraction of queries whose gold chunk is in the top 3 (i.e. what reaches the LLM)
* **MRR@10**: mean of 1/rank of the gold chunk inside the Top 10 (0 if missing)

In [14]:
EVAL_SET = [(q, gold) for _, _, q, gold, _ in FAILURE_QUERIES] + [
    # extra hard queries (kept out of FAILURE_QUERIES to save Gemini quota, still scored here)
    ("What penalty does a player get for an incorrect draw claim in rapid chess?", ["A.3"]),
    ("Are other players allowed to talk about a game in progress?", ["12.7"]),
    ("What happens if both flags have fallen and nobody knows which fell first?", ["III.3.1"]),
    ("How does the knight move?", ["3.6"]),
    ("Can a player who touches a piece always be forced to move it?", ["4.2.1"]),
    ("Does resigning always mean the player loses?", ["5.1.2"]),
    ("What happens after a player makes a second illegal move?", ["7.5.5"]),
    ("Is a player allowed to bring a mobile phone into the playing venue?", ["11.3.2", "12.8"]),
    ("When is a player not permitted to castle?", ["3.8.2.1", "3.8.2.2"]),
    ("Who is not allowed to adjust the pieces on the board?", ["4.2.1", "6.2.5"]),
    ("Can a player promote a pawn to a piece that has not been captured?", ["3.7.3.4"]),
    ("What happens if the game started with the wrong colours?", ["7.3"]),
    ("What happens if a player arrives late to the game?", ["6.7.1"]),
    ("Can a player withdraw a draw offer?", ["9.1.2.1"]),
    ("Is smoking allowed during the game?", ["11.3.4"]),
    ("Can a player write down their move before playing it?", ["8.1.2"]),
    ("What happens if a player touches the king then the rook but castling is illegal?", ["4.4.3"]),
    ("Who must give permission for a player to leave the playing venue?", ["11.2.3"]),
    ("In blitz, how many times can a player ask to see the scoresheet?", ["B.2.3"]),
    ("How does a visually impaired player indicate that a piece is touched?", ["D.2.2"]),
    ("Can the arbiter search a player's body for devices?", ["11.3.3"]),
    ("When is a player excused from writing down moves?", ["8.4"]),
    ("What happens if a player claims threefold repetition incorrectly?", ["9.5.3"]),
    ("Can both players lose the same game?", ["11.8"]),
    ("Is a score of three quarters to one quarter allowed?", ["10.2"]),
    ("Is the game a draw if a player runs out of time but the opponent only has a king?", ["6.9"]),
    ("Can a player take back a move after letting go of the piece?", ["4.7"]),
    ("Can a player castle if their king is currently in check?", ["3.8.2.2"]),
    ("Is en passant allowed two moves after the pawn advanced?", ["3.7.3.2"]),
]

def evaluate(eval_set):
    rows = []
    for q, gold in eval_set:
        cands, _ = retrieve_top_k(q)
        reranked, _ = rerank(q, [dict(c) for c in cands])
        g = gold_chunk_ids(gold)
        br, cr = best_rank(reranked, g, "bi_rank"), best_rank(reranked, g, "ce_rank")
        rows.append({"query": q, "gold": ", ".join(gold), "baseline rank": br, "re-ranked rank": cr})
    return pd.DataFrame(rows)

eval_df = evaluate(EVAL_SET)
hit3  = lambda s: np.mean([x is not None and not pd.isna(x) and x <= TOP_N for x in s])
mrr   = lambda s: np.mean([0 if x is None or pd.isna(x) else 1 / x for x in s])
recall10 = np.mean(eval_df["baseline rank"].notna())

metrics_df = pd.DataFrame({
    "metric": ["Hit@3 (gold reaches the LLM)", "MRR@10", "Recall@10 (gold in candidates, same for both)"],
    "Pipeline 1: baseline": [hit3(eval_df["baseline rank"]), mrr(eval_df["baseline rank"]), recall10],
    "Pipeline 2: re-rank":  [hit3(eval_df["re-ranked rank"]), mrr(eval_df["re-ranked rank"]), recall10],
}).round(3)
display(metrics_df)

def outcome(row):
    b, c = row["baseline rank"], row["re-ranked rank"]
    if pd.isna(b): return "missed by both (not in top 10)"
    if b > TOP_N and c <= TOP_N: return "fixed by re-ranking"
    if b <= TOP_N and c > TOP_N: return "broken by re-ranking"
    return "top 3 in both" if b <= TOP_N else "outside top 3 in both"
eval_df["outcome"] = eval_df.apply(outcome, axis=1)
display(eval_df["outcome"].value_counts().to_frame("queries"))
display(eval_df)

,metric,Pipeline 1: baseline,Pipeline 2: re-rank
0,Hit@3 (gold reaches the LLM),0.611,0.889
1,MRR@10,0.567,0.718
2,"Recall@10 (gold in candidates, same for both)",0.972,0.972


,queries
outcome,
top 3 in both,20
fixed by re-ranking,12
broken by re-ranking,2
outside top 3 in both,1
missed by both (not in top 10),1


,query,gold,baseline rank,re-ranked rank,outcome
0,Can a player stop the clock to go to the toilet?,6.11.2,8.0,1.0,fixed by re-ranking
1,Can a player use two hands to castle?,7.5.4,8.0,1.0,fixed by re-ranking
2,"If a player runs out of time, do they always lose the game?",6.9,7.0,2.0,fixed by re-ranking
3,Is the draw automatic after 50 moves without capture?,9.6.2,4.0,3.0,fixed by re-ranking
4,What is the arbiter not allowed to tell a player during the game?,12.6,4.0,3.0,fixed by re-ranking
5,Can a player be penalised for knocking pieces over?,7.4.3,8.0,3.0,fixed by re-ranking
6,Can the time on the clock be corrected after an illegal move?,7.1,3.0,7.0,broken by re-ranking
7,What penalty does a player get for an incorrect draw claim in rapid chess?,A.3,4.0,2.0,fixed by re-ranking
8,Are other players allowed to talk about a game in progress?,12.7,4.0,1.0,fixed by re-ranking
9,What happens if both flags have fallen and nobody knows which fell first?,III.3.1,5.0,1.0,fixed by re-ranking


## 12. Latency profiling

Each stage is timed separately with `time.perf_counter()` (CPU only, no GPU):

* **Stage 1**: query embedding + FAISS search (same for both pipelines)
* **Stage 2**: cross-encoder scoring of 10 pairs (re-rank pipeline only)
* **LLM**: Gemini API call (network-bound; measured in Section 10)

Retrieval stages are repeated 5× over all evaluation queries for stable numbers.

In [16]:
REPEATS = 5
t_stage1, t_stage2 = [], []
for _ in range(REPEATS):
    for q, _ in EVAL_SET:
        cands, t1 = retrieve_top_k(q);      t_stage1.append(t1)
        _, t2 = rerank(q, [dict(c) for c in cands]); t_stage2.append(t2)

ms = lambda xs: np.array(xs) * 1000
s1, s2 = ms(t_stage1), ms(t_stage2)
llm_b = ms([runs[k][0]["t_llm"] for k in runs]); llm_r = ms([runs[k][1]["t_llm"] for k in runs])

def stats(name, arr):
    return {"stage": name, "mean ms": arr.mean(), "median ms": np.median(arr),
            "p95 ms": np.percentile(arr, 95), "n": len(arr)}

latency_df = pd.DataFrame([
    stats("Stage 1: embed query + FAISS top-10", s1),
    stats("Stage 2: cross-encoder on 10 pairs", s2),
    stats("Retrieval total — Pipeline 1 (baseline)", s1),
    stats("Retrieval total — Pipeline 2 (re-rank)", s1 + s2),
    stats("LLM generation — Pipeline 1", llm_b),
    stats("LLM generation — Pipeline 2", llm_r),
    stats("End-to-end — Pipeline 1", np.median(s1) + llm_b),
    stats("End-to-end — Pipeline 2", np.median(s1 + s2) + llm_r),
]).round(1)
display(latency_df)
print(f"Re-ranking adds ~{np.median(s2):.1f} ms per query "
      f"({np.median(s1 + s2) / np.median(s1):.1f}x the retrieval time), "
      f"while the LLM call is ~{np.median(np.concatenate([llm_b, llm_r])):.0f} ms.")

,stage,mean ms,median ms,p95 ms,n
0,Stage 1: embed query + FAISS top-10,18.9,17.5,26.8,180
1,Stage 2: cross-encoder on 10 pairs,825.4,770.6,1169.9,180
2,Retrieval total — Pipeline 1 (baseline),18.9,17.5,26.8,180
3,Retrieval total — Pipeline 2 (re-rank),844.3,787.2,1196.1,180
4,LLM generation — Pipeline 1,4281.9,2812.2,10321.6,7
5,LLM generation — Pipeline 2,3184.1,2119.0,7574.2,7
6,End-to-end — Pipeline 1,4299.4,2829.7,10339.1,7
7,End-to-end — Pipeline 2,3971.3,2906.3,8361.4,7


Re-ranking adds ~770.6 ms per query (44.9x the retrieval time), while the LLM call is ~2537 ms.


## 13. Scaling experiment: why not cross-encode the whole corpus?

The bi-encoder encodes chunks **once, offline**; at query time it only embeds the query and does a
vector search. The cross-encoder must run a full transformer pass for **every (query, chunk) pair at query time**.
Below we time the cross-encoder on 10, 25, 50, and all chunks of this document and extrapolate linearly to
larger collections.

In [17]:
q = FAILURE_QUERIES[0][2]
scale_rows = []
for n in [10, 25, 50, len(chunks)]:
    pairs = [(q, c["text"]) for c in chunks[:n]]
    times = []
    for _ in range(5):
        t0 = time.perf_counter(); cross_encoder.predict(pairs, batch_size=32); times.append(time.perf_counter() - t0)
    scale_rows.append({"chunks scored by cross-encoder": n, "median ms": np.median(times) * 1000})
scale_df = pd.DataFrame(scale_rows)
per_pair_ms = scale_df.iloc[-1]["median ms"] / len(chunks)

# Bi-encoder search over the full index, for comparison
_, t_full = retrieve_top_k(q, k=len(chunks))
extrap = pd.DataFrame([{
    "collection size": f"{n:,} chunks",
    "cross-encoder on all chunks (est.)": f"{per_pair_ms * n / 1000:,.1f} s",
    "bi-encoder + FAISS (measured here / typical ANN)": f"{np.median(s1):.1f} ms" if n <= 1000 else "~10–50 ms with ANN index",
} for n in [len(chunks), 1_000, 100_000, 10_000_000]])

display(scale_df.round(1))
print(f"≈ {per_pair_ms:.2f} ms per (query, chunk) pair on this CPU")
display(extrap)

,chunks scored by cross-encoder,median ms
0,10,688.4
1,25,1842.2
2,50,4188.7
3,124,13754.1


≈ 110.92 ms per (query, chunk) pair on this CPU


,collection size,cross-encoder on all chunks (est.),bi-encoder + FAISS (measured here / typical ANN)
0,124 chunks,13.8 s,17.5 ms
1,"1,000 chunks",110.9 s,17.5 ms
2,"100,000 chunks","11,092.0 s",~10–50 ms with ANN index
3,"10,000,000 chunks","1,109,205.0 s",~10–50 ms with ANN index


## 14. Save results
All tables are written to `results/` so they can be used in the report.

In [18]:
chunk_df.to_csv(RESULTS_DIR / "chunks.csv", index=False)
shift_df.to_csv(RESULTS_DIR / "rank_shifts.csv", index=False)
answers_df.to_csv(RESULTS_DIR / "answers.csv", index=False)
eval_df.to_csv(RESULTS_DIR / "eval_ranks.csv", index=False)
metrics_df.to_csv(RESULTS_DIR / "retrieval_metrics.csv", index=False)
latency_df.to_csv(RESULTS_DIR / "latency.csv", index=False)
scale_df.to_csv(RESULTS_DIR / "cross_encoder_scaling.csv", index=False)

# Answers as readable markdown as well
with open(RESULTS_DIR / "answers.md", "w") as f:
    for row in answer_rows:
        f.write(f"## {row['id']} · {row['category']}\n**Q:** {row['query']}  \n**Gold clause:** {row['gold clause']}\n\n"
                f"**Answer 1 (baseline, context {row['baseline context']}):**\n\n{row['answer_1_baseline']}\n\n"
                f"**Answer 2 (re-rank, context {row['re-ranked context']}):**\n\n{row['answer_2_rerank']}\n\n---\n\n")

print("Saved:", sorted(p.name for p in RESULTS_DIR.iterdir()))

# One archive to download from a Colab runtime (Colab: Download...)
import shutil
shutil.make_archive("results", "zip", RESULTS_DIR)
print("Wrote results.zip")

Saved: ['answers.csv', 'answers.md', 'chunks.csv', 'cross_encoder_scaling.csv', 'eval_ranks.csv', 'latency.csv', 'llm_cache.json', 'rank_shifts.csv', 'retrieval_metrics.csv']
Wrote results.zip
